# Sec 2d — Forecast (multi-step) group metrics

Q1 (z-as-neural) — Y (ECoG) + Z (LFP) forecast quality vs horizon. PSID / DPAD / VARMA.
Q2 (z-as-behavior) — Y (ECoG) + Z (kinematics) forecast quality vs horizon.

Chapter plan figures:
* fig_q1_forecast_decay — Y + Z forecast accuracy vs horizon (quantile 0.10-0.90 band, pools all channels)
* fig_q1_forecast_decomp — Signal decomposition at 0.5s horizon: amplitude r / inst-freq r / phase PLV
* fig_q2_forecast_decay — Behavioral Y + Z forecast accuracy vs horizon
* fig_q2_forecast_decomp — Signal decomposition at 0.5s horizon (behavioral Y + Z)

Appendix figures:
* fig_081  — Pooled DBS raincloud, Y (ECoG) forecast, z-as-neural
* fig_081b — Pooled DBS raincloud, Z (LFP) forecast, z-as-neural
* fig_082  — 4-slot YZ raincloud: Y-neural / Z-neural / Y-behavioral / Z-behavioral
* fig_084  — Pooled DBS raincloud, Y (ECoG) forecast, z-as-behavior
* fig_084b — Pooled DBS raincloud, Z (kinematics) forecast, z-as-behavior
* fig_086  — Pooled forecast vs horizon, single channel (N)RMSE + Pearson r, SEM bands
* fig_087  — Raw vs env per-session, Z (LFP) forecast, z-as-neural
* fig_088  — Raw vs env per-session, Y (ECoG) forecast, z-as-behavior
* fig_088b — Raw vs env per-session, Y (ECoG) forecast, z-as-neural
* fig_088c — Raw vs env per-session, Z (kinematics) forecast, z-as-behavior
* fig_089  — Per-feature bar, Z (LFP) forecast, z-as-neural
* fig_089b — Per-feature bar, Y (ECoG) forecast, z-as-neural
* fig_089c — Per-feature bar, Y (ECoG) forecast, z-as-behavior
* fig_089d — Per-feature bar, Z (kinematics) forecast, z-as-behavior
* fig_091  — Hilbert amplitude r + PLV per-session, Z (LFP) forecast, z-as-neural
* fig_091b — Hilbert amplitude r + PLV per-session, Y (ECoG) forecast, z-as-neural
* fig_092  — Hilbert amplitude r + PLV per-session, Y (ECoG) forecast, z-as-behavior
* fig_092b — Hilbert amplitude r + PLV per-session, Z (kinematics) forecast, z-as-behavior
* fig_093  — Horizon all channels (N)RMSE + Pearson r, LFP Z + ECoG Y
* fig_094  — Band-grouped forecast quality (raw r / env r / PLV), LFP Z + ECoG Y
* fig_095  — Hilbert horizon decomposition (amplitude r / PLV / inst-freq error vs horizon)

In [ ]:
import sys, os

os.chdir("/home/bobby/repos/latent-neural-dynamics-modeling")
sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")

from pathlib import Path

OUT = Path("thesis_figures/sec2")
OUT.mkdir(parents=True, exist_ok=True)
results_root = Path("results").resolve()

import numpy as np
import matplotlib.pyplot as plt
import polars as pl
from matplotlib.patches import Patch

from modules.style import (
    apply_thesis_style,
    panel_label,
    COLOR_PSID,
    COLOR_DPAD,
    COLOR_VARMA,
    COLOR_DBS_OFF,
    COLOR_DBS_ON,
)
from modules.loaders import (
    EXP_Z_AS_BEHAVIOR,
    EXP_Z_AS_NEURAL,
    SESSIONS,
    load_split_results_required,
    resolve_neural_y_channel_idx,
)
from modules.utils import normalize_stim, trial_metric_forecast_for_model
from modules.sec2_common import (
    MODELS,
    MODEL_COLS,
    BAND_ORDER,
    CH_TYPES,
    DECOMP_METRICS,
    SAMPLING_HZ,
    SessionObj,
    make_session_obj,
    load_channel_names,
    load_forecast,
    collect_per_cell_metric,
    collect_decomp_metrics,
    collect_rawenv_metrics_per_session,
    collect_hilbert_metrics_per_session,
    collect_band_grouped_metrics,
    collect_per_feature_means,
    collect_horizon_all_channels,
    hilbert_horizon_collect,
    hilbert_per_step,
    window_decay,
    collect_decomp_at_horizon,
    per_step_rmse_and_pearson,
    horizon_quantile_bands,
    pool_dbs_cells,
    pool_yz_dbs_cells,
    raincloud_slot,
    dbs_raincloud_fig,
    yz_raincloud_fig,
    decomp_fig,
    band_raincloud_fig,
    hilbert_raincloud_sessions_fig,
    rawenv_raincloud_sessions_fig,
    per_feature_bar_fig,
    horizon_decay_fig,
)


def forecast_loader(variant, run_ts, split):
    res = load_forecast(results_root, variant, split)
    if "Z_future_true" in res:
        res["Z"] = res["Z_future_true"]
        res["Zp"] = res["Z_future_pred"]
    if "Y_future_true" in res:
        res["Y"] = res["Y_future_true"]
        res["Yp"] = res["Y_future_pred"]
    return res


score_z_fc = lambda res, i, ch, m: trial_metric_forecast_for_model(
    res, i, ch, m, forecast_target="Z"
)
score_y_fc = lambda res, i, ch, m: trial_metric_forecast_for_model(
    res, i, ch, m, forecast_target="Y"
)

all_session_objs = [
    make_session_obj(results_root, s, EXP_Z_AS_BEHAVIOR) for s in SESSIONS
]
all_lap_session_objs = [
    make_session_obj(results_root, s, EXP_Z_AS_NEURAL) for s in SESSIONS
]

apply_thesis_style()
print("imports OK")

## Appendix — Distribution alternatives + drift (pooled Y forecast, RMSE)

Companion to sec2c distribution analysis but on forecast trajectories.
Per-trial RMSE on the pooled neural Y channel; four lenses:
- raincloud
- ECDF
- block-running median curve (12 chronological bins)
- train->val->test cascade with paired-median arrows

## Appendix — Per-cell & pooled forecast overview (mirror of sec2c figures)

Per-cell box plots (Y + Z forecast metrics) + pooled rainclouds (Y, Z).
Laplacian z-as-neural and kinematics z-as-behavior.
All three frameworks: PSID, DPAD, VARMA — forecast heads produce Y_future_pred / Z_future_pred.

## Appendix — figs_081/081b/082: Pooled DBS + YZ rainclouds (forecast, z-as-neural)

Raincloud: per-trial RMSE/Pearson r pooled across sessions, DBS-OFF vs DBS-ON.
fig_081: Y (ECoG) self-forecast. fig_081b: Z (LFP) forecast. fig_082: 4-slot YZ combined.

In [ ]:
lap_y_r_fc = collect_per_cell_metric(
    all_lap_session_objs, forecast_loader, score_y_fc, "Y", "pearson", "test"
)
lap_y_n_fc = collect_per_cell_metric(
    all_lap_session_objs, forecast_loader, score_y_fc, "Y", "rmse", "test"
)
lap_z_r_fc = collect_per_cell_metric(
    all_lap_session_objs, forecast_loader, score_z_fc, "Z", "pearson", "test"
)
lap_z_n_fc = collect_per_cell_metric(
    all_lap_session_objs, forecast_loader, score_z_fc, "Z", "rmse", "test"
)

In [ ]:
lap_y_r_fc_pool = pool_dbs_cells(lap_y_r_fc, SESSIONS)
lap_y_n_fc_pool = pool_dbs_cells(lap_y_n_fc, SESSIONS)
fig = dbs_raincloud_fig(lap_y_r_fc_pool, lap_y_n_fc_pool)
fig.savefig(str(OUT / "fig_081_lap_pool_raincloud_y_forecast.png"))
plt.show()

In [ ]:
# Fig 081b — Pooled DBS raincloud, Z (Laplacian LFP) forecast, z-as-neural
lap_z_r_fc_pool = pool_dbs_cells(lap_z_r_fc, SESSIONS)
lap_z_n_fc_pool = pool_dbs_cells(lap_z_n_fc, SESSIONS)
fig = dbs_raincloud_fig(lap_z_r_fc_pool, lap_z_n_fc_pool)
fig.savefig(str(OUT / "fig_081b_lap_pool_raincloud_z_fc.png"))
plt.show()

In [ ]:
# 4-slot YZ figure: Y-neural / Z-neural / Y-behavioral / Z-behavioral
neu_yz_r = pool_yz_dbs_cells(
    lap_y_r_fc, lap_z_r_fc, SESSIONS, y_label="Y-neural", z_label="Z-neural"
)
neu_yz_n = pool_yz_dbs_cells(
    lap_y_n_fc, lap_z_n_fc, SESSIONS, y_label="Y-neural", z_label="Z-neural"
)
beh_yz_r = pool_yz_dbs_cells(
    beh_y_r_fc, beh_z_r_fc, SESSIONS, y_label="Y-behavioral", z_label="Z-behavioral"
)
beh_yz_n = pool_yz_dbs_cells(
    beh_y_n_fc, beh_z_n_fc, SESSIONS, y_label="Y-behavioral", z_label="Z-behavioral"
)
fig = yz_raincloud_fig({**neu_yz_r, **beh_yz_r}, {**neu_yz_n, **beh_yz_n})
fig.savefig(str(OUT / "fig_082_pool_raincloud_yz4_forecast.png"))
plt.show()

## Appendix — figs_084/084b: Pooled DBS rainclouds (forecast, z-as-behavior)

Per-trial RMSE/Pearson r, pooled across sessions, DBS-OFF vs DBS-ON.
fig_084: Y (ECoG) self-forecast. fig_084b: Z (kinematics) forecast.

In [ ]:
beh_y_r_fc = collect_per_cell_metric(
    all_session_objs, forecast_loader, score_y_fc, "Y", "pearson", "test"
)
beh_y_n_fc = collect_per_cell_metric(
    all_session_objs, forecast_loader, score_y_fc, "Y", "rmse", "test"
)
beh_z_r_fc = collect_per_cell_metric(
    all_session_objs, forecast_loader, score_z_fc, "Z", "pearson", "test"
)
beh_z_n_fc = collect_per_cell_metric(
    all_session_objs, forecast_loader, score_z_fc, "Z", "rmse", "test"
)

In [ ]:
beh_y_r_fc_pool = pool_dbs_cells(beh_y_r_fc, SESSIONS)
beh_y_n_fc_pool = pool_dbs_cells(beh_y_n_fc, SESSIONS)
fig = dbs_raincloud_fig(beh_y_r_fc_pool, beh_y_n_fc_pool)
fig.savefig(str(OUT / "fig_084_beh_pool_raincloud_y_forecast.png"))
plt.show()

In [ ]:
# Fig 084b — Pooled DBS raincloud, Z (tracing kinematics) forecast
beh_z_r_fc_pool = pool_dbs_cells(beh_z_r_fc, SESSIONS)
beh_z_n_fc_pool = pool_dbs_cells(beh_z_n_fc, SESSIONS)
fig = dbs_raincloud_fig(beh_z_r_fc_pool, beh_z_n_fc_pool)
fig.savefig(str(OUT / "fig_084b_beh_pool_raincloud_z_fc.png"))
plt.show()

## Appendix — fig_086: Single-channel forecast vs horizon (ECoG, z-as-behavior)

How forecast quality decays with horizon for a single ECoG channel.
Top: NRMSE (z-scored per trial). Bottom: Pearson r across trials.
Lines: model hue (PSID/DPAD/VARMA). Solid=DBS-OFF, dashed=DBS-ON. Band = ±2 SEM.

In [ ]:
def collect_horizon_per_step(
    triplets,
    *,
    target="Y",
    channel_idx=0,
    neural_y_feature_name="ECOG_1_beta_27_30_raw",
    split="test"
):
    """Per (model, dbs), pool (n_trials, n_steps) true/pred for a single channel."""
    from modules.lib.transforms import reshape_future_z_time_first

    k_true = "Y_future_true" if target == "Y" else "Z_future_true"
    k_pred = "Y_future_pred" if target == "Y" else "Z_future_pred"
    out = {
        (m, d): {"true": [], "pred": []}
        for m in ("PSID", "DPAD", "VARMA")
        for d in ("off", "on")
    }
    for tri in triplets:
        for model_name, variant, run_ts in (
            ("PSID", tri.psid_variant, tri.psid_run_ts),
            ("DPAD", tri.dpad_variant, tri.dpad_run_ts),
            ("VARMA", tri.varma_variant, tri.varma_run_ts),
        ):
            if not variant or not run_ts:
                continue
            res = forecast_loader(variant, run_ts, split)
            arr_t = res.get(k_true, [])
            arr_p = res.get(k_pred, [])
            if not arr_t or not arr_p:
                continue
            if target == "Y":
                try:
                    ch_use = resolve_neural_y_channel_idx(
                        res, neural_y_feature_name, channel_idx
                    )
                except (ValueError, KeyError):
                    ch_use = int(channel_idx)
            else:
                ch_use = int(channel_idx)
            for i in range(len(arr_t)):
                if arr_t[i] is None or arr_p[i] is None:
                    continue
                stim = normalize_stim(res["stim"][i])
                if stim is None:
                    continue
                try:
                    T = reshape_future_z_time_first(np.asarray(arr_t[i], dtype=float))
                    P = reshape_future_z_time_first(np.asarray(arr_p[i], dtype=float))
                except (ValueError, Exception):
                    continue
                if T.shape != P.shape or ch_use >= T.shape[1]:
                    continue
                t_vec, p_vec = T[:, ch_use], P[:, ch_use]
                msk = np.isfinite(t_vec) & np.isfinite(p_vec)
                if not msk.all():
                    continue
                mu = float(np.mean(t_vec))
                sigma = float(np.std(t_vec)) or 1.0
                out[(model_name, stim)]["true"].append((t_vec - mu) / sigma)
                out[(model_name, stim)]["pred"].append((p_vec - mu) / sigma)
    result = {}
    for k, d in out.items():
        if not d["true"]:
            continue
        m_min = min(t.size for t in d["true"])
        if m_min == 0:
            continue
        result[k] = (
            np.stack([t[:m_min] for t in d["true"]]),
            np.stack([p[:m_min] for p in d["pred"]]),
        )
    return result

In [ ]:
horizon_arrays = collect_horizon_per_step(
    all_session_objs,
    target="Y",
    channel_idx=5,
    neural_y_feature_name="ECOG_1_beta_27_30_raw",
)

fig, axes = plt.subplots(2, 1, figsize=(7.5, 8.0), sharex=True)
ax_rmse, ax_r = axes
panel_label(ax_rmse, "A", "ECoG - NRMSE")
panel_label(ax_r, "B", "ECoG - Pearson r")
ax_rmse.axhline(1.0, color="black", linestyle=":", linewidth=0.6, alpha=0.30, zorder=0)
ax_r.axhline(0.0, color="black", linestyle=":", linewidth=0.6, alpha=0.30, zorder=0)
dbs_style = {"off": "-", "on": "--"}
for (m, d), (T, P) in sorted(horizon_arrays.items()):
    rmse, sem, rs = per_step_rmse_and_pearson(T, P)
    x = np.arange(len(rs)) / SAMPLING_HZ * 1000
    c = MODEL_COLS[m]
    ls = dbs_style[d]
    ax_rmse.fill_between(x, rmse - 2 * sem, rmse + 2 * sem, color=c, alpha=0.12, lw=0)
    ax_rmse.plot(x, rmse, color=c, ls=ls, lw=1.4, label=f"{m} {d.upper()}")
    ax_r.plot(x, rs, color=c, ls=ls, lw=1.4)
ax_rmse.legend(loc="upper left", frameon=False, fontsize=8)
ax_r.set_xlabel("Horizon (ms)")
ax_rmse.set_ylabel("NRMSE")
ax_r.set_ylabel("Pearson r")
fig.savefig(str(OUT / "fig_086_single_channel_forecast.png"))
plt.show()

## Appendix — figs_087-093: Forecast channel analysis (mirror of sec2c appendix)

Raw vs env breakdown (per-session raincloud), per-channel median bar, and
100ms-window time-decay — on forecast outputs (Z_future / Y_future).

## Q1 Neural — Y + Z forecast vs horizon [PRIMARY]

Chapter plan: fig_q1_forecast_decay, fig_q1_forecast_decomp.
Pools all channels across sessions. Quantile 0.10-0.90 band.

In [ ]:
print("Computing LFP Z forecast horizon (all channels)...")
q1_fc_hz = collect_horizon_all_channels(
    all_lap_session_objs, forecast_loader, target="Z"
)
fig = horizon_decay_fig(q1_fc_hz, "LFP Z - Pearson r", "LFP Z - NRMSE")
fig.savefig(str(OUT / "fig_q1_forecast_decay.png"))
plt.show()

## fig_q1_forecast_decomp — Signal decomposition at 0.5s horizon (Y + Z)

Apply Hilbert decomp on the first 0.5s of the forecast trajectory for each (trial, channel).
3 rows (amplitude r / inst-freq r / phase PLV) x 3 cols (PSID / DPAD / VARMA).

In [ ]:
print("Computing decomp metrics at 0.5s horizon: LFP Z...")
q1_fc_decomp = collect_decomp_at_horizon(q1_fc_hz, horizon_ms=500.0)
fig = decomp_fig(q1_fc_decomp)
fig.savefig(str(OUT / "fig_q1_forecast_decomp.png"))
plt.show()

## Q2 Behavioral — Y + Z forecast vs horizon [PRIMARY]

Same structure as Q1 but Y = ECoG, Z = tracing kinematics (velocity_x, acceleration_magnitude).
Chapter plan: fig_q2_forecast_decay, fig_q2_forecast_decomp.

In [ ]:
print("Computing behavioral Z forecast horizon (all channels)...")
q2_fc_hz = collect_horizon_all_channels(all_session_objs, forecast_loader, target="Z")
fig = horizon_decay_fig(q2_fc_hz, "Behavioral Z - Pearson r", "Behavioral Z - NRMSE")
fig.savefig(str(OUT / "fig_q2_forecast_decay.png"))
plt.show()

## fig_q2_forecast_decomp — Signal decomposition at 0.5s horizon (behavioral Y + Z)

Same 3x3 structure as fig_q1_forecast_decomp. Y = ECoG, Z = tracing kinematics.

In [ ]:
print("Computing decomp metrics at 0.5s horizon: behavioral Z...")
q2_fc_decomp = collect_decomp_at_horizon(q2_fc_hz, horizon_ms=500.0)
fig = decomp_fig(q2_fc_decomp)
fig.savefig(str(OUT / "fig_q2_forecast_decomp.png"))
plt.show()

In [ ]:
ch_z_neu = lambda s: load_channel_names(results_root, s, EXP_Z_AS_NEURAL, "Z_features")
lap_z_rawenv_r_ps = collect_rawenv_metrics_per_session(
    all_lap_session_objs,
    EXP_Z_AS_NEURAL,
    "Z",
    "pearson",
    "test",
    forecast_loader,
    score_z_fc,
    ch_z_neu,
)
lap_z_rawenv_n_ps = collect_rawenv_metrics_per_session(
    all_lap_session_objs,
    EXP_Z_AS_NEURAL,
    "Z",
    "rmse",
    "test",
    forecast_loader,
    score_z_fc,
    ch_z_neu,
)

fig = rawenv_raincloud_sessions_fig(lap_z_rawenv_r_ps, lap_z_rawenv_n_ps, SESSIONS)
fig.savefig(str(OUT / "fig_087_lap_rawenv_z_forecast_per_session.png"))
plt.show()

In [ ]:
ch_y_beh = lambda s: load_channel_names(
    results_root, s, EXP_Z_AS_BEHAVIOR, "Y_features"
)
beh_y_rawenv_r_ps = collect_rawenv_metrics_per_session(
    all_session_objs,
    EXP_Z_AS_BEHAVIOR,
    "Y",
    "pearson",
    "test",
    forecast_loader,
    score_y_fc,
    ch_y_beh,
)
beh_y_rawenv_n_ps = collect_rawenv_metrics_per_session(
    all_session_objs,
    EXP_Z_AS_BEHAVIOR,
    "Y",
    "rmse",
    "test",
    forecast_loader,
    score_y_fc,
    ch_y_beh,
)

fig = rawenv_raincloud_sessions_fig(beh_y_rawenv_r_ps, beh_y_rawenv_n_ps, SESSIONS)
fig.savefig(str(OUT / "fig_088_beh_rawenv_y_forecast_per_session.png"))
plt.show()

In [ ]:
# Fig 088b — Raw vs envelope, Y (ECoG) forecast, z-as-neural
ch_y_neu = lambda s: load_channel_names(results_root, s, EXP_Z_AS_NEURAL, "Y_features")
lap_y_rawenv_r_ps = collect_rawenv_metrics_per_session(
    all_lap_session_objs,
    EXP_Z_AS_NEURAL,
    "Y",
    "pearson",
    "test",
    forecast_loader,
    score_y_fc,
    ch_y_neu,
)
lap_y_rawenv_n_ps = collect_rawenv_metrics_per_session(
    all_lap_session_objs,
    EXP_Z_AS_NEURAL,
    "Y",
    "rmse",
    "test",
    forecast_loader,
    score_y_fc,
    ch_y_neu,
)

fig = rawenv_raincloud_sessions_fig(lap_y_rawenv_r_ps, lap_y_rawenv_n_ps, SESSIONS)
fig.savefig(str(OUT / "fig_088b_lap_rawenv_y_forecast_per_session.png"))
plt.show()

In [ ]:
# Fig 088c — Raw vs envelope, Z (tracing kinematics) forecast, z-as-behavior
ch_z_beh = lambda s: load_channel_names(
    results_root, s, EXP_Z_AS_BEHAVIOR, "Z_features"
)
beh_z_rawenv_r_ps = collect_rawenv_metrics_per_session(
    all_session_objs,
    EXP_Z_AS_BEHAVIOR,
    "Z",
    "pearson",
    "test",
    forecast_loader,
    score_z_fc,
    ch_z_beh,
)
beh_z_rawenv_n_ps = collect_rawenv_metrics_per_session(
    all_session_objs,
    EXP_Z_AS_BEHAVIOR,
    "Z",
    "rmse",
    "test",
    forecast_loader,
    score_z_fc,
    ch_z_beh,
)

fig = rawenv_raincloud_sessions_fig(beh_z_rawenv_r_ps, beh_z_rawenv_n_ps, SESSIONS)
fig.savefig(str(OUT / "fig_088c_beh_rawenv_z_forecast_per_session.png"))
plt.show()

In [ ]:
# Fig 089 — Per-feature bar, Z (Laplacian LFP) forecast, z-as-neural
z_neural_means_fc, z_neural_feats_fc = collect_per_feature_means(
    all_lap_session_objs,
    EXP_Z_AS_NEURAL,
    "Z",
    "pearson",
    "test",
    forecast_loader,
    score_z_fc,
    lambda s: load_channel_names(results_root, s, EXP_Z_AS_NEURAL, "Z_features"),
)
fig = per_feature_bar_fig(z_neural_means_fc, z_neural_feats_fc, SESSIONS)
fig.savefig(str(OUT / "fig_089_z_neural_perfeature_fc.png"))
plt.show()

In [ ]:
# Fig 089b — Per-feature bar, Y (ECoG) forecast, z-as-neural
y_neural_means_fc, y_neural_feats_fc = collect_per_feature_means(
    all_lap_session_objs,
    EXP_Z_AS_NEURAL,
    "Y",
    "pearson",
    "test",
    forecast_loader,
    score_y_fc,
    lambda s: load_channel_names(results_root, s, EXP_Z_AS_NEURAL, "Y_features"),
)
fig = per_feature_bar_fig(y_neural_means_fc, y_neural_feats_fc, SESSIONS)
fig.savefig(str(OUT / "fig_089b_y_neural_perfeature_fc.png"))
plt.show()

In [ ]:
# Fig 089c — Per-feature bar, Y (ECoG) forecast, tracing kinematics mode
y_kin_means_fc, y_kin_feats_fc = collect_per_feature_means(
    all_session_objs,
    EXP_Z_AS_BEHAVIOR,
    "Y",
    "pearson",
    "test",
    forecast_loader,
    score_y_fc,
    lambda s: load_channel_names(results_root, s, EXP_Z_AS_BEHAVIOR, "Y_features"),
)
fig = per_feature_bar_fig(y_kin_means_fc, y_kin_feats_fc, SESSIONS)
fig.savefig(str(OUT / "fig_089c_y_kin_perfeature_fc.png"))
plt.show()

In [ ]:
# Fig 089d — Per-feature bar, Z (tracing kinematics) forecast, z-as-behavior
z_kin_means_fc, z_kin_feats_fc = collect_per_feature_means(
    all_session_objs,
    EXP_Z_AS_BEHAVIOR,
    "Z",
    "pearson",
    "test",
    forecast_loader,
    score_z_fc,
    lambda s: load_channel_names(results_root, s, EXP_Z_AS_BEHAVIOR, "Z_features"),
)
fig = per_feature_bar_fig(z_kin_means_fc, z_kin_feats_fc, SESSIONS)
fig.savefig(str(OUT / "fig_089d_z_kin_perfeature_fc.png"))
plt.show()

In [ ]:
ch_z_neu = lambda s: load_channel_names(results_root, s, EXP_Z_AS_NEURAL, "Z_features")
lap_z_hilbert_ps_fc = collect_hilbert_metrics_per_session(
    all_lap_session_objs, EXP_Z_AS_NEURAL, "Z", "test", forecast_loader, ch_z_neu
)

fig = hilbert_raincloud_sessions_fig(lap_z_hilbert_ps_fc, SESSIONS)
fig.savefig(str(OUT / "fig_091_lap_hilbert_z_forecast_per_session.png"))
plt.show()

In [ ]:
ch_y_beh = lambda s: load_channel_names(
    results_root, s, EXP_Z_AS_BEHAVIOR, "Y_features"
)
beh_y_hilbert_ps_fc = collect_hilbert_metrics_per_session(
    all_session_objs, EXP_Z_AS_BEHAVIOR, "Y", "test", forecast_loader, ch_y_beh
)

fig = hilbert_raincloud_sessions_fig(beh_y_hilbert_ps_fc, SESSIONS)
fig.savefig(str(OUT / "fig_092_beh_hilbert_y_forecast_per_session.png"))
plt.show()

In [ ]:
# Fig 091b — Hilbert amplitude r + PLV, Y (ECoG) forecast, z-as-neural
ch_y_neu = lambda s: load_channel_names(results_root, s, EXP_Z_AS_NEURAL, "Y_features")
neural_y_hilbert_ps_fc = collect_hilbert_metrics_per_session(
    all_lap_session_objs, EXP_Z_AS_NEURAL, "Y", "test", forecast_loader, ch_y_neu
)

fig = hilbert_raincloud_sessions_fig(neural_y_hilbert_ps_fc, SESSIONS)
fig.savefig(str(OUT / "fig_091b_neural_hilbert_y_fc_per_session.png"))
plt.show()

In [ ]:
# Fig 092b — Hilbert amplitude r + PLV, Z (tracing kinematics) forecast, z-as-behavior
ch_z_beh = lambda s: load_channel_names(
    results_root, s, EXP_Z_AS_BEHAVIOR, "Z_features"
)
beh_z_hilbert_ps_fc = collect_hilbert_metrics_per_session(
    all_session_objs, EXP_Z_AS_BEHAVIOR, "Z", "test", forecast_loader, ch_z_beh
)

fig = hilbert_raincloud_sessions_fig(beh_z_hilbert_ps_fc, SESSIONS)
fig.savefig(str(OUT / "fig_092b_beh_hilbert_z_forecast_per_session.png"))
plt.show()

## Appendix — fig_094: Band-grouped forecast quality (raw r / env r / PLV), Z + Y

Mirror of sec2c fig_076 but on forecast trajectories. Rows: Laplacian LFP (Z) / ECoG (Y).
Columns: oscillatory r / amplitude envelope r / phase PLV. Per-band, per-model raincloud.

In [ ]:
print("Computing band-grouped forecast metrics...")
ch_z_neu = lambda s: load_channel_names(results_root, s, EXP_Z_AS_NEURAL, "Z_features")
ch_y_neu = lambda s: load_channel_names(results_root, s, EXP_Z_AS_NEURAL, "Y_features")
ch_z_beh = lambda s: load_channel_names(
    results_root, s, EXP_Z_AS_BEHAVIOR, "Z_features"
)
ch_y_beh = lambda s: load_channel_names(
    results_root, s, EXP_Z_AS_BEHAVIOR, "Y_features"
)

lap_fc_raw_z, lap_fc_env_z, lap_fc_plv_z = collect_band_grouped_metrics(
    all_lap_session_objs,
    EXP_Z_AS_NEURAL,
    "Z",
    "test",
    forecast_loader,
    score_z_fc,
    ch_z_neu,
)
lap_fc_raw_y, lap_fc_env_y, lap_fc_plv_y = collect_band_grouped_metrics(
    all_lap_session_objs,
    EXP_Z_AS_NEURAL,
    "Y",
    "test",
    forecast_loader,
    score_y_fc,
    ch_y_neu,
)
beh_fc_raw_z, beh_fc_env_z, beh_fc_plv_z = collect_band_grouped_metrics(
    all_session_objs,
    EXP_Z_AS_BEHAVIOR,
    "Z",
    "test",
    forecast_loader,
    score_z_fc,
    ch_z_beh,
)
beh_fc_raw_y, beh_fc_env_y, beh_fc_plv_y = collect_band_grouped_metrics(
    all_session_objs,
    EXP_Z_AS_BEHAVIOR,
    "Y",
    "test",
    forecast_loader,
    score_y_fc,
    ch_y_beh,
)

fig, axes = plt.subplots(2, 3, figsize=(12, 7.0), layout="constrained")
band_raincloud_fig(axes[0, 0], lap_fc_raw_z, "r - raw", legend=True)
band_raincloud_fig(axes[0, 1], lap_fc_env_z, "r - env")
band_raincloud_fig(axes[0, 2], lap_fc_plv_z, "PLV")
band_raincloud_fig(axes[1, 0], beh_fc_raw_y, "r - raw")
band_raincloud_fig(axes[1, 1], beh_fc_env_y, "r - env")
band_raincloud_fig(axes[1, 2], beh_fc_plv_y, "PLV")
panel_label(axes[0, 0], "A", "LFP Z - raw")
panel_label(axes[0, 1], "B", "LFP Z - env")
panel_label(axes[0, 2], "C", "LFP Z - PLV")
panel_label(axes[1, 0], "D", "ECoG Y - raw")
panel_label(axes[1, 1], "E", "ECoG Y - env")
panel_label(axes[1, 2], "F", "ECoG Y - PLV")
fig.savefig(str(OUT / "fig_094_band_grouped_forecast.png"))
plt.show()

## Appendix — fig_095: Hilbert horizon decomposition (amplitude r / PLV / inst-freq vs horizon)

3x2 layout: rows = amplitude r / phase PLV / inst-freq error; cols = Laplacian LFP (Z) / ECoG (Y).
x-axis = forecast horizon (ms). Lines per model (PSID/DPAD/VARMA), solid=DBS-OFF, dashed=DBS-ON.
Inst-freq from d(unwrap(phase))/dt; error = mean |f_true - f_pred| across trials (Hz).

In [ ]:
print("Computing Hilbert horizon data...")
hh_z = hilbert_horizon_collect(all_lap_session_objs, forecast_loader, target="Z")
hh_y = hilbert_horizon_collect(all_session_objs, forecast_loader, target="Y")

dbs_style = {"off": "-", "on": "--"}
fig, axes = plt.subplots(3, 2, figsize=(10, 8.0), layout="constrained", sharex="col")
panel_label(axes[0, 0], "A", "Laplacian LFP - amplitude r")
panel_label(axes[0, 1], "B", "ECoG - amplitude r")
panel_label(axes[1, 0], "C", "Laplacian LFP - PLV")
panel_label(axes[1, 1], "D", "ECoG - PLV")
panel_label(axes[2, 0], "E", "Laplacian LFP - inst-freq err")
panel_label(axes[2, 1], "F", "ECoG - inst-freq err")

for col, hh in enumerate([hh_z, hh_y]):
    for (m, d), (AT, AP, PT, PP) in sorted(hh.items()):
        ar, plv, fe = hilbert_per_step(AT, AP, PT, PP)
        x = np.arange(len(ar)) / SAMPLING_HZ * 1000
        c = MODEL_COLS[m]
        ls = dbs_style[d]
        axes[0, col].plot(x, ar, color=c, ls=ls, lw=1.2, label=f"{m} {d.upper()}")
        axes[1, col].plot(x, plv, color=c, ls=ls, lw=1.2)
        axes[2, col].plot(x, fe, color=c, ls=ls, lw=1.2)

for ax in axes.flatten():
    ax.set_xlabel("Horizon (ms)")
axes[0, 0].legend(loc="upper right", frameon=False, fontsize=7)
fig.savefig(str(OUT / "fig_095_hilbert_horizon.png"))
plt.show()

## Appendix — fig_093: Forecast horizon (N)RMSE + Pearson r, all channels, LFP Z + ECoG Y

Per-step RMSE (z-scored) and Pearson r vs horizon, pooled across all channels and trials.
2x2 layout: (Pearson r, NRMSE) x (LFP Z, ECoG Y). Lines per model per DBS state.

In [ ]:
lap_z_horizon = collect_horizon_all_channels(
    all_lap_session_objs, forecast_loader, target="Z"
)
beh_y_horizon = collect_horizon_all_channels(
    all_session_objs, forecast_loader, target="Y"
)

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
(ax_zr, ax_yr), (ax_zn, ax_yn) = axes
panel_label(ax_zr, "A", "Laplacian LFP - Pearson r")
panel_label(ax_yr, "B", "ECoG - Pearson r")
panel_label(ax_zn, "C", "Laplacian LFP - NRMSE")
panel_label(ax_yn, "D", "ECoG - NRMSE")

dbs_style = {"off": "-", "on": "--"}
for (m, d), (T, P) in sorted(lap_z_horizon.items()):
    rmse, sem, rs = per_step_rmse_and_pearson(T, P)
    x = np.arange(len(rs)) / SAMPLING_HZ * 1000
    c = MODEL_COLS[m]
    ls = dbs_style[d]
    ax_zr.plot(x, rs, color=c, ls=ls, lw=1.2, label=f"{m} {d.upper()}")
    ax_zn.plot(x, rmse, color=c, ls=ls, lw=1.2)

for (m, d), (T, P) in sorted(beh_y_horizon.items()):
    rmse, sem, rs = per_step_rmse_and_pearson(T, P)
    x = np.arange(len(rs)) / SAMPLING_HZ * 1000
    c = MODEL_COLS[m]
    ls = dbs_style[d]
    ax_yr.plot(x, rs, color=c, ls=ls, lw=1.2)
    ax_yn.plot(x, rmse, color=c, ls=ls, lw=1.2)

ax_zr.legend(loc="upper right", frameon=False, fontsize=7)
for ax in axes.flatten():
    ax.set_xlabel("Horizon (ms)")
fig.savefig(str(OUT / "fig_093_horizon_all_channels.png"))
plt.show()